In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import os

json_path = '/content/drive/MyDrive/analys_masks.json'
def reformat_json_for_processing(filepath):
    if not os.path.exists(filepath):
        print(f"Error: Could not find {filepath}")
        return
    with open(filepath, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    formatted_data = []

    for filename, stringified_json in raw_data.items():
        try:
            record = json.loads(stringified_json)
            record["filename"] = filename

            formatted_data.append(record)
        except json.JSONDecodeError:
            print(f"Could not parse the JSON string for {filename}")

    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(formatted_data, f, ensure_ascii=False, indent=4)

    print(f"Successfully reformatted {len(formatted_data)} records into a list of objects.")

reformat_json_for_processing(json_path)

In [ ]:
import json
import os

json_path = '/content/drive/MyDrive/analys_masks.json'

def process_part_numbers(filepath):
    if not os.path.exists(filepath):
        print(f"Error: Could not find {filepath}")
        return

    print(f"Loading data from {filepath}...")
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    processed_count = 0
    for record in data:
        pn = record.get("part_number")

        if pn and isinstance(pn, str) and "-" in pn:
            try:
                prefix, suffix = pn.split("-", 1)
                padded_suffix = suffix.zfill(4)[-4:]

                record["edited_pn"] = f"{prefix}-{padded_suffix}"
                processed_count += 1

            except ValueError:
                record["edited_pn"] = None
        else:
            record["edited_pn"] = None

    print(f"Formatted {processed_count} part numbers. Saving...")
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print("✅ Done! The 'edited_pn' field has been added and strictly formatted to 4 suffix digits.")

process_part_numbers(json_path)

In [ ]:
!pip install rapidfuzz

In [ ]:
import pandas as pd
import json
import os
from rapidfuzz import process, fuzz

excel_path = '/content/drive/MyDrive/analys_masks/Garrett.xlsx'
json_path = '/content/drive/MyDrive/analys_masks.json'
output_json = '/content/drive/MyDrive/analys_masks/FinalMatchedJson.json'

CONFIDENCE_THRESHOLD = 1.0

VISUAL_SETS = [
    {'0', '8', 'O', 'B', 'Q', 'D'},
    {'1', '7', 'I', 'L', 'T', '|', '/'},
    {'2', 'Z', '7'},
    {'5', 'S', '6'},
    {'6', 'G', '8'},
    {'4', 'A', 'H'},
    {'9', 'g', 'q', '0'}
]

def calculate_visual_score(extracted, candidate):
    """Applies a tiny penalty for known visual OCR mistakes, and a full penalty for others."""
    extracted = str(extracted).upper()
    candidate = str(candidate).upper()

    if len(extracted) != len(candidate):
        return fuzz.ratio(extracted, candidate)

    score = 100.0
    standard_penalty = 100.0 / len(extracted)
    visual_penalty = standard_penalty * 0.2

    for c1, c2 in zip(extracted, candidate):
        if c1 != c2:
            is_visual_mixup = any(c1 in v_set and c2 in v_set for v_set in VISUAL_SETS)
            score -= visual_penalty if is_visual_mixup else standard_penalty

    return max(0.0, score)

def get_best_visual_match(extracted_str, db_list):
    """Casts a wide net with RapidFuzz, then re-ranks the top 10 visually."""
    if not extracted_str or str(extracted_str).strip() == "":
        return None, 0.0

    top_candidates = process.extract(extracted_str, db_list, scorer=fuzz.ratio, limit=10)

    best_str = None
    best_score = 0.0

    for candidate_str, _, _ in top_candidates:
        v_score = calculate_visual_score(extracted_str, candidate_str)
        if v_score > best_score:
            best_score = v_score
            best_str = candidate_str

    return best_str, best_score

def match_edited_pn():
    print(f"Loading database from {excel_path}...")
    df = pd.read_excel(excel_path, usecols=[0])
    db_pns = df.iloc[:, 0].dropna().astype(str).str.strip().tolist()
    print(f"Loading JSON from {json_path}...")
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    print("Matching 'edited_pn' values using Visual Similarity...")
    match_count = 0

    for record in data:
        extracted_pn = record.get("edited_pn")

        matched_string, score = get_best_visual_match(extracted_pn, db_pns)

        if matched_string and score >= CONFIDENCE_THRESHOLD:
            record["db_match"] = matched_string
            record["match_score"] = round(score, 2)
            match_count += 1
        else:
            record["db_match"] = "NO_STRONG_MATCH" if extracted_pn else None
            record["match_score"] = round(score, 2) if extracted_pn else 0.0

    os.makedirs(os.path.dirname(output_json), exist_ok=True)
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"✅ Successfully matched {match_count} items. Saved to {output_json}")

if __name__ == "__main__":
    match_edited_pn()

In [ ]:
import json
import os

gt_json_path = '/content/drive/MyDrive/analys_masks/GarretGT.json'
pred_json_path = '/content/drive/MyDrive/analys_masks/FinalMatchedJson.json'

def evaluate_predictions():
    print(f"Loading Ground Truth from {gt_json_path}...")
    with open(gt_json_path, 'r', encoding='utf-8') as f:
        gt_data = json.load(f)
    print(f"Loading Predictions from {pred_json_path}...")
    with open(pred_json_path, 'r', encoding='utf-8') as f:
        pred_data = json.load(f)

    gt_map = {}
    for key, value in gt_data.items():
        file_name = os.path.basename(key)
        base_name = os.path.splitext(file_name)[0]
        gt_map[base_name] = value.get("edited_pn")

    total_evaluated = 0
    correct_matches = 0
    mismatches = []
    missing_in_gt = []

    for item in pred_data:
        pred_image = item.get("image_name", "")
        base_name = pred_image.replace("_cutout", "")

        base_name = base_name.replace("mask_", "")
        base_name = os.path.splitext(base_name)[0]

        if base_name in gt_map:
            total_evaluated += 1
            gt_pn = gt_map[base_name]
            pred_pn = item.get("db_match")

            if pred_pn == gt_pn:
                correct_matches += 1
            else:
                mismatches.append({
                    "image": base_name,
                    "predicted": pred_pn,
                    "ground_truth": gt_pn,
                    "match_score": item.get("match_score")
                })
        else:
            missing_in_gt.append(base_name)

    print("\n" + "="*40)
    print("           EVALUATION RESULTS           ")
    print("="*40)
    print(f"Total files in Predictions: {len(pred_data)}")
    print(f"Total overlapping with GT:  {total_evaluated}")
    print(f"Correct Matches:            {correct_matches}")

    if total_evaluated > 0:
        accuracy = (correct_matches / total_evaluated) * 100
        print(f"Accuracy:                   {accuracy:.2f}%\n")
    else:
        print("Accuracy:                   N/A (No overlapping files found)\n")

    if mismatches:
        print("--- Mismatched Items ---")
        for m in mismatches:
            print(f"Image: {m['image']:<12} | Pred: {str(m['predicted']):<15} | GT: {str(m['ground_truth']):<15} | Score: {m['match_score']}")

    if missing_in_gt:
        print(f"\n[!] Note: {len(missing_in_gt)} predicted images were not found in the Ground Truth file:")
        for missing_img in missing_in_gt:
            print(f"  - {missing_img}")

if __name__ == "__main__":
    evaluate_predictions()